# EV Autonomous Driving System — Demo

This notebook is a small, synthetic demonstration. It is not a measured vehicle stack and it is not an ASIL-D safety case.

## What you are looking at

- Full PointNet++ in `src/perception/pointnet.py`: 512 points, then 128, then a global 1024-d feature. The cell below is a short walkthrough of that stack's per-point features.
- Context-conditioned attention. The scenario id is a learned embedding, not a language model.
- Missing sensors use a known device and a padding mask. They do not read an uninitialized tensor.
- Timing below is a single forward pass. Percentile measurement is `src/inference/benchmark.py`.

The closed loop (sync, Isaac stand-in, tracker, stage times) runs with `python benchmarks/run_feedback_loop.py`.

## 📁 Directory Structure
```
src/
├── perception/          # 3D perception algorithms
│   ├── pointnet.py     # PointNet++ implementation
│   ├── feature_extractor.py  # Feature extraction
│   ├── object_detection.py   # 3D object detection
│   ├── segmentation.py       # Semantic segmentation
│   └── perception_system.py  # Main perception pipeline
└── fusion/             # Multi-modal sensor fusion
    ├── sensor_fusion.py      # Sensor data fusion
    ├── llm_attention.py      # LLM-guided attention
    ├── temporal_fusion.py    # Temporal processing
    └── multi_modal_fusion.py # Main fusion pipeline
```


## 🔧 Setup and Imports


In [ ]:
# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple
import asyncio
import time
from loguru import logger

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ Setup complete!")


# 🎯 Perception System Demo

## Full PointNet++

Production set abstraction lives in `src/perception/pointnet.py`: farthest-point sampling, ball query, and levels at 512, 128, and global. This cell shows the per-point feature path those levels pool over.


In [ ]:
# Simplified PointNet++ implementation for demo
class PointNetPlusPlus(nn.Module):
    """Walkthrough head. The full 512/128/global PointNet++ is src/perception/pointnet.py."""
    
    def __init__(self, num_classes=20, num_features=64):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = num_features
        
        # Simplified architecture for demo
        self.conv1 = nn.Conv1d(num_features, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)
        
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, points, features=None):
        """Forward pass through PointNet++."""
        if features is None:
            features = points.transpose(1, 2)
        
        # Feature extraction
        x = F.relu(self.conv1(features))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        
        # Global max pooling
        x = torch.max(x, dim=2)[0]
        
        # Classification
        output = self.classifier(x)
        return output

# Production hierarchy: src/perception/pointnet.py (512 / 128 / global).
pointnet = PointNetPlusPlus(num_classes=20, num_features=3).to(device)
print(f"PointNet++ walkthrough parameters: {sum(p.numel() for p in pointnet.parameters())}")


In [ ]:
# Generate sample point cloud data
def generate_sample_pointcloud(num_points=1024):
    """Generate synthetic point cloud data for demonstration."""
    # Simulate LiDAR point cloud
    points = torch.randn(1, num_points, 3).to(device)  # [batch, points, xyz]
    
    # Add some structure (simulate ground plane and objects)
    points[0, :num_points//2, 2] = 0  # Ground plane
    points[0, num_points//2:, 2] = torch.randn(num_points//2).to(device) * 0.5 + 1  # Objects
    
    return points

# Generate and process point cloud
sample_points = generate_sample_pointcloud(1024)
print(f"📊 Generated point cloud: {sample_points.shape}")

# Process through PointNet++
with torch.no_grad():
    start_time = time.time()
    pointnet_output = pointnet(sample_points)
    processing_time = (time.time() - start_time) * 1000  # Convert to ms

print(f"⚡ PointNet++ processing time: {processing_time:.2f}ms")
print(f"🎯 Output shape: {pointnet_output.shape}")
print(f"📈 Class probabilities: {F.softmax(pointnet_output, dim=1).cpu().numpy()[0][:5]}...")


# 🔗 Multi-Modal Fusion System Demo

## Context-conditioned attention

Camera, lidar, and radar features are projected into one embedding. A scenario id adds a learned vector. No language model is called. A missing modality is a zero token plus a padding mask.


In [ ]:
# LLM-Guided Attention Implementation
class ContextConditionedAttention(nn.Module):
    """Shared embedding and masked attention. The scenario table is not an LLM."""
    
    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.sensor_types = ['lidar', 'camera', 'radar']
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        # LLM context embedding
        self.context_embedding = nn.Embedding(10, embed_dim)  # 10 different scenarios
        
        # Sensor-specific projections
        self.sensor_projections = nn.ModuleDict({
            'lidar': nn.Linear(1024, embed_dim),
            'camera': nn.Linear(2048, embed_dim),
            'radar': nn.Linear(512, embed_dim)
        })
    
    def forward(self, sensor_data):
        """Compute attention weights for sensor fusion."""
        device = next(self.parameters()).device
        present = [sensor_data[name] for name in self.sensor_types if sensor_data.get(name) is not None]
        if not present:
            raise ValueError('at least one modality is required')
        batch = present[0].shape[0]
        projected_features = []
        invalid = []
        for sensor_type in self.sensor_types:
            raw = sensor_data.get(sensor_type)
            if raw is None:
                projected_features.append(torch.zeros(batch, self.embed_dim, device=device))
                invalid.append(True)
            else:
                projected_features.append(self.sensor_projections[sensor_type](raw))
                invalid.append(False)
        
        sensor_features = torch.stack(projected_features, dim=1)
        key_padding = torch.tensor(invalid, device=device).unsqueeze(0).expand(batch, -1)
        attended_features, attention_weights = self.attention(
            sensor_features, sensor_features, sensor_features,
            key_padding_mask=key_padding,
        )
        
        # Get LLM context (simplified)
        scenario_id = torch.tensor([0]).to(device)  # Default scenario
        context = self.context_embedding(scenario_id)
        
        # Apply context guidance
        guided_features = attended_features + context.unsqueeze(1)
        
        return guided_features, attention_weights

llm_attention = ContextConditionedAttention(embed_dim=512, num_heads=8).to(device)
print("Context-conditioned attention initialized")


In [ ]:
# Generate multi-modal sensor data
def generate_sensor_data():
    """Generate synthetic multi-modal sensor data."""
    sensor_data = {
        'lidar': torch.randn(1, 1024).to(device),      # LiDAR features
        'camera': torch.randn(1, 2048).to(device),     # Camera features  
        'radar': torch.randn(1, 512).to(device)        # Radar features
    }
    return sensor_data

# Process multi-modal data
sensor_data = generate_sensor_data()
print("📡 Generated multi-modal sensor data:")
for sensor_type, features in sensor_data.items():
    print(f"  {sensor_type}: {features.shape}")

# Apply LLM-guided attention
with torch.no_grad():
    start_time = time.time()
    fused_features, attention_weights = llm_attention(sensor_data)
    fusion_time = (time.time() - start_time) * 1000

print(f"\n⚡ Fusion processing time: {fusion_time:.2f}ms")
print(f"🔗 Fused features shape: {fused_features.shape}")
print(f"🎯 Attention weights shape: {attention_weights.shape}")
print(f"📊 Attention distribution: {attention_weights[0].cpu().numpy()}")


## 📊 Performance Visualization


In [ ]:
# Performance visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Processing times
processing_times = [processing_time, fusion_time]
labels = ['Point encoder', 'Fusion']
colors = ['#FF6B6B', '#4ECDC4']

axes[0, 0].bar(labels, processing_times, color=colors)
axes[0, 0].set_title('⚡ Processing Times', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Time (ms)')
axes[0, 0].axhline(y=32, color='red', linestyle='--', label='32ms target (not a measured vehicle budget)')
axes[0, 0].legend()

# 2. Attention weights heatmap
attention_matrix = attention_weights[0].cpu().numpy()
im = axes[0, 1].imshow(attention_matrix, cmap='Blues', aspect='auto')
axes[0, 1].set_title('Attention weights', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Sensor Index')
axes[0, 1].set_ylabel('Sensor Index')
axes[0, 1].set_xticks(range(3))
axes[0, 1].set_xticklabels(['LiDAR', 'Camera', 'Radar'])
axes[0, 1].set_yticks(range(3))
axes[0, 1].set_yticklabels(['LiDAR', 'Camera', 'Radar'])
plt.colorbar(im, ax=axes[0, 1])

# 3. Class probabilities
class_probs = F.softmax(pointnet_output, dim=1)[0].cpu().numpy()
class_names = [f'Class {i}' for i in range(len(class_probs))]
axes[1, 0].bar(range(len(class_probs)), class_probs, color='skyblue')
axes[1, 0].set_title('Class scores from an untrained encoder', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Probability')
axes[1, 0].set_xticks(range(0, len(class_probs), 2))

# 4. System architecture overview
axes[1, 1].text(0.5, 0.8, '🚗 EV Autonomous Driving System', 
                ha='center', va='center', fontsize=16, fontweight='bold')
axes[1, 1].text(0.5, 0.6, 'Perception Pipeline:', 
                ha='center', va='center', fontsize=12, fontweight='bold')
axes[1, 1].text(0.5, 0.5, 'PointNet++ → Feature Extraction → Object Detection', 
                ha='center', va='center', fontsize=10)
axes[1, 1].text(0.5, 0.4, 'Fusion Pipeline:', 
                ha='center', va='center', fontsize=12, fontweight='bold')
axes[1, 1].text(0.5, 0.3, 'Sensors → masked attention → tracker', 
                ha='center', va='center', fontsize=10)
axes[1, 1].text(0.5, 0.1, f'Total Latency: {processing_time + fusion_time:.2f}ms', 
                ha='center', va='center', fontsize=12, color='green', fontweight='bold')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"Single-pass notebook time: {processing_time + fusion_time:.2f} ms")
print('This is not a p50/p95 vehicle measurement. See src/inference/benchmark.py.')


## Safety boundary

`src/safety` explores timeouts, fault flags, and WCET notes. That is a prototype. It is not an ISO 26262 ASIL-D certification, a hazard analysis, or a proof of a failure rate.

## Conclusion

1. Full PointNet++ is `src/perception/pointnet.py`, with set abstraction at 512, 128, and global.
2. Fusion here is masked attention conditioned on a scenario id.
3. One forward pass is not a latency budget.
4. Run `python benchmarks/run_feedback_loop.py` for sync, the Isaac stand-in, tracking, and percentile timing.
